In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("debug-bronze-read")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [8]:
from pyspark.sql.functions import *

metrics = spark.read.parquet("s3a://gold/data_quality_metrics/")
metrics.toPandas()
# metrics.filter(col("status") == "FAIL").orderBy("run_timestamp").show(truncate=False)

,run_id,run_timestamp,pipeline,layer,check_name,expected,actual,status,details
0,20260511_053850,2026-05-11 05:41:58.510045,retail_orders,silver,anomaly_zero_revenue_months,0,0,PASS,Months with zero revenue
1,20260511_053850,2026-05-11 05:42:00.617134,retail_orders,silver,anomaly_monthly_revenue_deviation,within 30% of mean (£779513.36),3,WARN,"Anomalous months: 2010-10: £1122889.07, 2010-1..."
2,20260511_053850,2026-05-11 05:41:49.331108,retail_orders,silver,business_rule_event_type_values,0,0,PASS,Rows with unexpected event_type values
3,20260511_053850,2026-05-11 05:41:50.938324,retail_orders,silver,referential_integrity_customer_id,0,103,WARN,103 orders have customer_id not found in custo...
4,20260511_053850,2026-05-11 05:41:37.421545,retail_orders,silver,null_check_event_time,<=0.0%,0.0%,PASS,0 nulls out of 518446
5,20260511_053850,2026-05-11 05:41:38.031624,retail_orders,silver,null_check_event_type,<=0.0%,0.0%,PASS,0 nulls out of 518446
6,20260511_053850,2026-05-11 05:41:40.385276,retail_customers,silver,null_check_first_seen_time,<=0.0%,0.0%,PASS,0 nulls out of 4382
7,20260511_053850,2026-05-11 05:41:46.560300,retail_orders,silver,duplicate_check,0,0,PASS,Duplicates remaining after dedup on (invoice_i...
8,20260511_053850,2026-05-11 05:41:47.641704,retail_customers,silver,duplicate_check,0,0,PASS,Duplicate customer_ids in silver
9,20260511_053850,2026-05-11 05:41:48.146150,retail_orders,silver,business_rule_zero_quantity,0,0,PASS,Rows where quantity = 0


In [4]:
orders_bronze = spark.read.parquet("s3a://bronze/orders/")

In [6]:
customers_bronze = spark.read.parquet("s3a://bronze/customers/")

In [7]:
print("Orders bronze:", orders_bronze.count())
print("Customers bronze:", customers_bronze.count())

Orders bronze: 525425
Customers bronze: 4382


In [8]:
orders_bronze.printSchema()
customers_bronze.printSchema()

root
 |-- kafka_key: string (nullable = true)
 |-- raw_json: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- bronze_ingestion_time: timestamp (nullable = false)

root
 |-- kafka_key: string (nullable = true)
 |-- raw_json: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- bronze_ingestion_time: timestamp (nullable = false)



In [9]:
import json
sample = orders_bronze.select("raw_json").limit(1).collect()[0][0]
print(json.dumps(json.loads(sample), indent=2))

{
  "event_type": "order_item_created",
  "invoice_id": "489436",
  "stock_code": 22194,
  "description": "BLACK DINER WALL CLOCK",
  "quantity": 2,
  "unit_price": 8.5,
  "country": "United Kingdom",
  "customer_id": 13078,
  "event_time": "2009-12-01 09:06:00",
  "total_amount": 17.0,
  "ingestion_time": "2026-04-07 12:23:02.478111"
}


In [10]:
sample_c = customers_bronze.select("raw_json").limit(1).collect()[0][0]
print(json.dumps(json.loads(sample_c), indent=2))

{
  "event_type": "customer_activity",
  "customer_id": 13085,
  "country": "United Kingdom",
  "first_seen_time": "2009-12-01 07:45:00",
  "ingestion_time": "2026-04-07 12:23:00.677787"
}


In [11]:
from pyspark.sql.functions import get_json_object

orders_bronze.select(
    get_json_object("raw_json", "$.event_type").alias("event_type")
).groupBy("event_type").count().show()

+--------------------+------+
|          event_type| count|
+--------------------+------+
|  order_item_created|511998|
|                NULL|  2928|
|order_item_cancelled| 10499|
+--------------------+------+



In [12]:
orders_bronze.filter(
    get_json_object("raw_json", "$.event_type").isNull()
).select(
    get_json_object("raw_json", "$.description").alias("description"),
    get_json_object("raw_json", "$.quantity").alias("quantity"),
    get_json_object("raw_json", "$.invoice_id").alias("invoice_id")
).show(5)

+-----------+--------+----------+
|description|quantity|invoice_id|
+-----------+--------+----------+
|       NULL|    NULL|      NULL|
|       NULL|    NULL|      NULL|
|       NULL|    NULL|      NULL|
|       NULL|    NULL|      NULL|
|       NULL|    NULL|      NULL|
+-----------+--------+----------+
only showing top 5 rows



In [13]:
orders_bronze.filter(
    get_json_object("raw_json", "$.event_type").isNull()
).select(
    get_json_object("raw_json", "$.quantity").cast("int").alias("quantity")
).describe().show()

+-------+--------+
|summary|quantity|
+-------+--------+
|  count|       0|
|   mean|    NULL|
| stddev|    NULL|
|    min|    NULL|
|    max|    NULL|
+-------+--------+



In [14]:
orders_bronze.filter(
    get_json_object("raw_json", "$.event_type").isNull()
).select("raw_json").limit(3).show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|raw_json                                                                                                                                                                                                                                                                                              |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"event_type": "order_item_cancelled", "invoice_id": "489521", "stock_code": 21646, "description": NaN, "qua

In [15]:
from pyspark.sql.functions import get_json_object as gjo, col

orders_parsed = orders_bronze.select(
    gjo("raw_json", "$.event_type").alias("event_type"),
    gjo("raw_json", "$.invoice_id").alias("invoice_id"),
    gjo("raw_json", "$.stock_code").alias("stock_code"),
    gjo("raw_json", "$.quantity").cast("int").alias("quantity"),
    gjo("raw_json", "$.unit_price").cast("double").alias("unit_price"),
    gjo("raw_json", "$.customer_id").alias("customer_id"),
    gjo("raw_json", "$.country").alias("country"),
    gjo("raw_json", "$.event_time").alias("event_time"),
    gjo("raw_json", "$.total_amount").cast("double").alias("total_amount"),
)

orders_parsed.printSchema()
orders_parsed.show(3)

root
 |-- event_type: string (nullable = true)
 |-- invoice_id: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- total_amount: double (nullable = true)

+------------------+----------+----------+--------+----------+-----------+--------------+-------------------+------------+
|        event_type|invoice_id|stock_code|quantity|unit_price|customer_id|       country|         event_time|total_amount|
+------------------+----------+----------+--------+----------+-----------+--------------+-------------------+------------+
|order_item_created|    489436|     22194|       2|       8.5|      13078|United Kingdom|2009-12-01 09:06:00|        17.0|
|order_item_created|    489436|    84596L|       8|      1.25|      13078|United Kingdom|2009-12-01 09:06:00|        10.0|
|o

In [16]:
from pyspark.sql.functions import col, sum as spark_sum

orders_parsed.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in orders_parsed.columns
]).show()

+----------+----------+----------+--------+----------+-----------+-------+----------+------------+
|event_type|invoice_id|stock_code|quantity|unit_price|customer_id|country|event_time|total_amount|
+----------+----------+----------+--------+----------+-----------+-------+----------+------------+
|      2928|      2928|      2928|    2928|      2928|     107927|   2928|      2928|        2928|
+----------+----------+----------+--------+----------+-----------+-------+----------+------------+



In [17]:
orders_parsed.filter(col("event_type").isNotNull()).groupBy("event_type").count().show()

+--------------------+------+
|          event_type| count|
+--------------------+------+
|  order_item_created|511998|
|order_item_cancelled| 10499|
+--------------------+------+



In [18]:
raw_dupes_created = 513135 - 511998
raw_dupes_cancelled = 12326 - 10499
print(raw_dupes_created, raw_dupes_cancelled)  # 1137, 1827
print(raw_dupes_created + raw_dupes_cancelled)  # = 2964 vs 2928 NULLs

1137 1827
2964


In [20]:
from pyspark.sql.functions import regexp_replace

orders_fixed = orders_bronze.withColumn(
    "raw_json_fixed",
    regexp_replace("raw_json", r':\s*NaN', ': null')
)

# Now parse from fixed json
orders_recovered = orders_fixed.filter(
    get_json_object("raw_json", "$.event_type").isNull()
).select(
    gjo("raw_json_fixed", "$.event_type").alias("event_type"),
    gjo("raw_json_fixed", "$.invoice_id").alias("invoice_id"),
    gjo("raw_json_fixed", "$.stock_code").alias("stock_code"),
    gjo("raw_json_fixed", "$.quantity").cast("int").alias("quantity"),
    gjo("raw_json_fixed", "$.unit_price").cast("double").alias("unit_price"),
    gjo("raw_json_fixed", "$.customer_id").alias("customer_id"),
    gjo("raw_json_fixed", "$.country").alias("country"),
    gjo("raw_json_fixed", "$.event_time").alias("event_time"),
    gjo("raw_json_fixed", "$.total_amount").cast("double").alias("total_amount"),
)

orders_recovered.show(3)
print(orders_recovered.count())

+--------------------+----------+----------+--------+----------+-----------+--------------+-------------------+------------+
|          event_type|invoice_id|stock_code|quantity|unit_price|customer_id|       country|         event_time|total_amount|
+--------------------+----------+----------+--------+----------+-----------+--------------+-------------------+------------+
|order_item_cancelled|    489521|     21646|      50|       0.0|       NULL|United Kingdom|2009-12-01 11:44:00|         0.0|
|order_item_cancelled|    489655|     20683|      44|       0.0|       NULL|United Kingdom|2009-12-01 17:26:00|         0.0|
|  order_item_created|    489659|     21350|     230|       0.0|       NULL|United Kingdom|2009-12-01 17:39:00|         0.0|
+--------------------+----------+----------+--------+----------+-----------+--------------+-------------------+------------+
only showing top 3 rows

2928


In [21]:
orders_recovered.groupBy("event_type").count().show()

+--------------------+-----+
|          event_type|count|
+--------------------+-----+
|  order_item_created| 1101|
|order_item_cancelled| 1827|
+--------------------+-----+



In [22]:
orders_bronze.select("offset").describe().show()

+-------+------------------+
|summary|            offset|
+-------+------------------+
|  count|            525425|
|   mean| 87595.78267688063|
| stddev|50603.491888110184|
|    min|                 0|
|    max|            179352|
+-------+------------------+



In [24]:
from pyspark.sql.functions import lag, col
from pyspark.sql.window import Window

w = Window.partitionBy("partition").orderBy("offset")

orders_bronze.withColumn(
    "prev_offset", lag("offset", 1).over(w)
).withColumn(
    "gap", col("offset") - col("prev_offset")
).filter(col("gap") > 1).select(
    "partition", "prev_offset", "offset", "gap"
).show()

+---------+-----------+------+---+
|partition|prev_offset|offset|gap|
+---------+-----------+------+---+
+---------+-----------+------+---+



In [ ]:
orders_bronze.select("partition").distinct().show()

In [16]:
orders_silver = spark.read.parquet("s3a://silver/orders/")
customers_silver = spark.read.parquet("s3a://silver/customers/")

print(orders_silver.count())
print(customers_silver.count())
orders_silver.printSchema()

518446
4382
root
 |-- kafka_key: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- bronze_ingestion_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- invoice_id: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- silver_ingestion_time: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [11]:
from pyspark.sql.types import *
orders_schema = StructType([
    StructField("event_type", StringType(), True),
    StructField("invoice_id", StringType(), True),
    StructField("stock_code", StringType(), True),
    StructField("description", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("country", StringType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("event_time", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("ingestion_time", StringType(), True)
])

In [12]:
# After parsing, before any filtering
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import *
orders_parsed_count = orders_bronze.withColumn(
    "raw_json", regexp_replace("raw_json", r':\s*NaN', ': null')
).withColumn(
    "parsed", from_json(col("raw_json"), orders_schema)
).select("parsed.*").count()

print("After parsing:", orders_parsed_count)

# How many have null event_time
null_event_time = orders_bronze.withColumn(
    "raw_json", regexp_replace("raw_json", r':\s*NaN', ': null')
).withColumn(
    "parsed", from_json(col("raw_json"), orders_schema)
).select("parsed.*").filter(col("event_time").isNull()).count()

print("Null event_time (dropped):", null_event_time)
print("Remaining after filter:", orders_parsed_count - null_event_time)
print("After dedup (silver):", 512093)
print("Dropped by dedup:", orders_parsed_count - null_event_time - 512093)

After parsing: 525425
Null event_time (dropped): 0
Remaining after filter: 525425
After dedup (silver): 512093
Dropped by dedup: 13332


In [14]:
# Check duplicates in bronze parsed data
from pyspark.sql.functions import col, sum as spark_sum
orders_bronze.withColumn(
    "raw_json", regexp_replace("raw_json", r':\s*NaN', ': null')
).withColumn(
    "parsed", from_json(col("raw_json"), orders_schema)
).select("parsed.*").withColumn(
    "event_time", to_timestamp("event_time")
).groupBy("invoice_id", "stock_code", "event_time") \
 .count() \
 .filter(col("count") > 1) \
 .agg(
     spark_sum(col("count") - 1).alias("total_duplicate_rows")
 ).show()

+--------------------+
|total_duplicate_rows|
+--------------------+
|               13332|
+--------------------+



In [15]:
# See what actual duplicates look like
orders_bronze.withColumn(
    "raw_json", regexp_replace("raw_json", r':\s*NaN', ': null')
).withColumn(
    "parsed", from_json(col("raw_json"), orders_schema)
).select("parsed.*").withColumn(
    "event_time", to_timestamp("event_time")
).groupBy("invoice_id", "stock_code", "event_time") \
 .count() \
 .filter(col("count") > 1) \
 .show(5)

+----------+----------+-------------------+-----+
|invoice_id|stock_code|         event_time|count|
+----------+----------+-------------------+-----+
|    491155|     22111|2009-12-10 09:49:00|    2|
|    489529|     22030|2009-12-01 11:51:00|    2|
|    489536|     21786|2009-12-01 12:13:00|    2|
|    489782|     20751|2009-12-02 11:45:00|    2|
|    489536|     21809|2009-12-01 12:13:00|    2|
+----------+----------+-------------------+-----+
only showing top 5 rows



In [17]:
# Count true duplicates in bronze
orders_bronze.withColumn(
    "raw_json", regexp_replace("raw_json", r':\s*NaN', ': null')
).withColumn(
    "parsed", from_json(col("raw_json"), orders_schema)
).select("parsed.*").withColumn(
    "event_time", to_timestamp("event_time")
).groupBy("invoice_id", "stock_code", "event_time", "quantity") \
 .count() \
 .filter(col("count") > 1) \
 .agg(spark_sum(col("count") - 1).alias("true_dupes_in_bronze")) \
 .show()

+--------------------+
|true_dupes_in_bronze|
+--------------------+
|                6979|
+--------------------+



In [19]:
orders_silver.filter(col("invoice_id") == "489517").toPandas()

,kafka_key,topic,partition,offset,kafka_timestamp,bronze_ingestion_time,event_type,invoice_id,stock_code,description,quantity,unit_price,country,customer_id,event_time,total_amount,ingestion_time,silver_ingestion_time,year,month
0,489517,retail_order_events,2,171,2026-04-07 12:23:28.263,2026-04-07 12:26:00.403,order_item_created,489517,20711,JUMBO BAG TOYS,1,1.95,United Kingdom,16329,2009-12-01 11:34:00,1.95,2026-04-07 12:23:28.263757,2026-04-25 10:04:05.481821,2009,12
1,489517,retail_order_events,2,146,2026-04-07 12:23:28.255,2026-04-07 12:26:00.403,order_item_created,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,1.95,United Kingdom,16329,2009-12-01 11:34:00,1.95,2026-04-07 12:23:28.255242,2026-04-25 10:04:05.481821,2009,12
2,489517,retail_order_events,2,160,2026-04-07 12:23:28.259,2026-04-07 12:26:00.403,order_item_created,489517,16207A,PINK STRAWBERRY HANDBAG,2,2.95,United Kingdom,16329,2009-12-01 11:34:00,5.90,2026-04-07 12:23:28.259177,2026-04-25 10:04:05.481821,2009,12
3,489517,retail_order_events,2,148,2026-04-07 12:23:28.255,2026-04-07 12:26:00.403,order_item_created,489517,21790,VINTAGE SNAP CARDS,1,0.85,United Kingdom,16329,2009-12-01 11:34:00,0.85,2026-04-07 12:23:28.255242,2026-04-25 10:04:05.481821,2009,12
4,489517,retail_order_events,2,147,2026-04-07 12:23:28.255,2026-04-07 12:26:00.403,order_item_created,489517,21791,VINTAGE HEADS AND TAILS CARD GAME,1,1.25,United Kingdom,16329,2009-12-01 11:34:00,1.25,2026-04-07 12:23:28.255242,2026-04-25 10:04:05.481821,2009,12
5,489517,retail_order_events,2,142,2026-04-07 12:23:28.253,2026-04-07 12:26:00.403,order_item_created,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,3.75,United Kingdom,16329,2009-12-01 11:34:00,3.75,2026-04-07 12:23:28.253230,2026-04-25 10:04:05.481821,2009,12
6,489517,retail_order_events,2,170,2026-04-07 12:23:28.260,2026-04-07 12:26:00.403,order_item_created,489517,21931,JUMBO STORAGE BAG SUKI,1,1.95,United Kingdom,16329,2009-12-01 11:34:00,1.95,2026-04-07 12:23:28.260955,2026-04-25 10:04:05.481821,2009,12
7,489517,retail_order_events,2,152,2026-04-07 12:23:28.258,2026-04-07 12:26:00.403,order_item_created,489517,20971,PINK BLUE FELT CRAFT TRINKET BOX,1,1.25,United Kingdom,16329,2009-12-01 11:34:00,1.25,2026-04-07 12:23:28.258152,2026-04-25 10:04:05.481821,2009,12
8,489517,retail_order_events,2,157,2026-04-07 12:23:28.259,2026-04-07 12:26:00.403,order_item_created,489517,21705,BAG 500g SWIRLY MARBLES,1,1.65,United Kingdom,16329,2009-12-01 11:34:00,1.65,2026-04-07 12:23:28.259177,2026-04-25 10:04:05.481821,2009,12
9,489517,retail_order_events,2,154,2026-04-07 12:23:28.258,2026-04-07 12:26:00.403,order_item_created,489517,22122,SET OF 2 FANCY FONT TEA TOWELS,12,2.95,United Kingdom,16329,2009-12-01 11:34:00,35.40,2026-04-07 12:23:28.258152,2026-04-25 10:04:05.481821,2009,12
